# DRL Trading System — Kaggle T4 GPU
Based on CLaC@FinMMEval2026 + your Qwen2.5-3B fine-tuned sentiment (90.42% F1)

### Before Running
1. Kaggle Secrets → `HF_TOKEN` | GPU T4 x1 | Internet ON
2. Run Cell 1 → **Restart** → Run All

In [ ]:
!pip install -q -U transformers peft datasets yfinance gymnasium scikit-learn torchao
import torch, transformers
print(f"Torch {torch.__version__} | CUDA {torch.cuda.is_available()} | Transformers {transformers.__version__}")
print("Restart kernel then Run All")

In [ ]:
# ── Config ────────────────────────────────────────────────────
ASSET      = "TSLA"            # or "BTC-USD"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_END  = "2022-12-31"; VAL_START = "2023-01-01"
VAL_END    = "2024-07-31"; TEST_START = "2024-08-01"
MAX_EP=1000; EVAL_EVERY=50; PATIENCE=10
REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
BASE = "Qwen/Qwen2.5-3B-Instruct"
print(f"Asset: {ASSET} | Device: {DEVICE}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("✅ HF login OK")

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
import yfinance as yf
import json, os, time, gc, copy, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

In [ ]:
def load_prices(ticker):
    df = yf.download(ticker, start="2010-01-01", auto_adjust=True, progress=False)
    df = df[["Close","Volume"]].copy()
    df.columns = ["close","volume"]
    df.index = pd.to_datetime(df.index).normalize()
    return df.dropna().reset_index().rename(columns={"Date":"date"})

prices = load_prices(ASSET)
prices["date"] = pd.to_datetime(prices["date"])
train_df = prices[prices["date"] <= TRAIN_END].copy()
val_df   = prices[(prices["date"] >= VAL_START) & (prices["date"] <= VAL_END)].copy()
test_df  = prices[prices["date"] >= TEST_START].copy()

for name, df in [("Train",train_df),("Val",val_df),("Test",test_df)]:
    print(f"{name}: {len(df)} days ({df['date'].min().date()} to {df['date'].max().date()})")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

print(f"Loading SEntFiN Qwen2.5-3B on {DEVICE}...")
sent_tok   = AutoTokenizer.from_pretrained(REPO)
sent_base  = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
sent_model = PeftModel.from_pretrained(sent_base, REPO)
sent_model.eval()

ASSET_NAME = {"TSLA":"Tesla (TSLA)","BTC-USD":"Bitcoin (BTC)"}.get(ASSET, ASSET)
SYSTEM = (f"You are a financial sentiment analyst. Classify the sentiment towards {ASSET_NAME}. "
          f"Reply with exactly one word: positive, negative, or neutral.")

lids={}
for lbl in ["positive","negative","neutral"]:
    ids=[]
    for v in [lbl," "+lbl,lbl.capitalize()," "+lbl.capitalize()]:
        ids.extend(sent_tok.encode(v,add_special_tokens=False))
    lids[lbl]=list(set(ids))
n_pos=len(lids["positive"]); n_neg=len(lids["negative"])
all_ids=lids["positive"]+lids["negative"]+lids["neutral"]

def score_article(text):
    msgs=[{"role":"system","content":SYSTEM},
          {"role":"user","content":f"Headline: {' '.join(text.split()[:300])}\nCompany: {ASSET_NAME}\nSentiment:"}]
    try: prompt=sent_tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    except: prompt=f"{SYSTEM}\n\nHeadline: {text[:300]}\nCompany: {ASSET_NAME}\nSentiment:"
    inp=sent_tok(prompt,return_tensors="pt",truncation=True,max_length=400).to(DEVICE)
    with torch.no_grad(): logits=sent_model(**inp).logits[0,-1,:]
    probs=torch.softmax(logits[all_ids],dim=0).cpu()
    return float(probs[:n_pos].sum()-probs[n_pos:n_pos+n_neg].sum())

print("✅ Scorer ready")
print(f"Test corporate action: {score_article('Infosys opens new AI lab with Nvidia'):.3f} (expect >0)")
print(f"Test negative:         {score_article('Company cuts annual revenue forecast'):.3f} (expect <0)")

In [ ]:
import gc
# Replace with real news data for best results
# Demo: random scores to test the pipeline end-to-end
print("Using demo sentiment — replace with real news articles for production")
np.random.seed(42)
train_sent=pd.Series(np.random.uniform(-0.3,0.3,len(train_df)).tolist(), index=train_df["date"].tolist())
val_sent  =pd.Series(np.random.uniform(-0.3,0.3,len(val_df)).tolist(),   index=val_df["date"].tolist())
test_sent =pd.Series(np.random.uniform(-0.3,0.3,len(test_df)).tolist(),  index=test_df["date"].tolist())
del sent_model, sent_base; gc.collect(); torch.cuda.empty_cache()
print("✅ Sentiment ready | GPU cleared")

In [ ]:
def build_states(df, sentiment, warmup=60):
    df = df.copy().sort_values("date").reset_index(drop=True)
    c, v = df["close"], df["volume"]
    ema10 = c.ewm(span=10,  adjust=False).mean()
    ema50 = c.ewm(span=50,  adjust=False).mean()
    delta = c.diff()
    gain  = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    rsi14 = 100 - (100 / (1 + gain/(loss+1e-10)))
    ema12 = c.ewm(span=12,  adjust=False).mean()
    ema26 = c.ewm(span=26,  adjust=False).mean()
    macd  = ema12 - ema26
    sig   = macd.ewm(span=9, adjust=False).mean()
    sma20 = c.rolling(20).mean()
    std20 = c.rolling(20).std()
    df["log_ret"]   = np.log(c/c.shift(1)+1e-10)
    df["daily_ret"] = c.pct_change()
    df["ema10_f"]   = ema10/c - 1
    df["ema50_f"]   = ema50/c - 1
    df["rsi_f"]     = rsi14/50 - 1
    df["macd_f"]    = (macd - sig)/(c+1e-10)
    df["boll_f"]    = (c - sma20)/(2*std20+1e-10)
    df["vol_f"]     = np.log(v/v.shift(1)+1e-10)
    dates = pd.DatetimeIndex(df["date"])
    df["dow_sin"] = np.sin(2*np.pi*dates.dayofweek/7)
    df["dow_cos"] = np.cos(2*np.pi*dates.dayofweek/7)
    df["dom_sin"] = np.sin(2*np.pi*dates.day/31)
    df["dom_cos"] = np.cos(2*np.pi*dates.day/31)
    df["moy_sin"] = np.sin(2*np.pi*dates.month/12)
    df["moy_cos"] = np.cos(2*np.pi*dates.month/12)
    df["sentiment"] = df["date"].map(
        dict(zip(sentiment.index, sentiment.values))
    ).fillna(0.0)
    df = df.iloc[warmup:].dropna().reset_index(drop=True)
    S = np.zeros((len(df), 15), dtype=np.float32)
    S[:,0]=df["log_ret"]; S[:,1]=0.0; S[:,2]=df["dow_sin"]; S[:,3]=df["dow_cos"]
    S[:,4]=df["dom_sin"]; S[:,5]=df["dom_cos"]; S[:,6]=df["moy_sin"]; S[:,7]=df["moy_cos"]
    S[:,8]=df["ema10_f"]; S[:,9]=df["ema50_f"]; S[:,10]=df["rsi_f"]
    S[:,11]=df["macd_f"]; S[:,12]=df["boll_f"]; S[:,13]=df["vol_f"]; S[:,14]=df["sentiment"]
    return np.clip(S,-10,10), df["daily_ret"].values.astype(np.float32)

WARMUP   = 60
val_ext  = pd.concat([train_df.tail(WARMUP), val_df]).reset_index(drop=True)
test_ext = pd.concat([val_df.tail(WARMUP), test_df]).reset_index(drop=True)
all_sent = pd.concat([train_sent, val_sent, test_sent])
train_S, train_R = build_states(train_df,  train_sent)
val_S,   val_R   = build_states(val_ext,   all_sent)
test_S,  test_R  = build_states(test_ext,  all_sent)
print(f"States: Train={len(train_S)} Val={len(val_S)} Test={len(test_S)}")

In [ ]:
class TradingEnv(gym.Env):
    """Alpha reward: r_t = log(V_t/V_{t-1}) - log(c_t/c_{t-1})  (paper Eq.3)"""
    P = {0:-1, 1:0, 2:1}
    def __init__(self, S, R, rand=True, cost=True):
        super().__init__()
        self.S=S.astype(np.float32); self.R=R.astype(np.float32)
        self.T=len(S); self.rand=rand; self.tc=0.002 if cost else 0.0
        self.action_space=gym.spaces.Discrete(3)
        self.observation_space=gym.spaces.Box(-np.inf,np.inf,(15,),np.float32)
        self.reset()
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t=int(np.random.randint(1,max(2,self.T-100))) if self.rand else 1
        self.val=100_000.0; self.prev=0; self.vals=[self.val]; self.acts=[0]
        return self._obs(), {}
    def _obs(self):
        s=self.S[self.t].copy(); s[1]=float(self.prev); return s
    def step(self, action):
        pos=self.P[action]; ret=float(self.R[self.t])
        cost=self.tc if pos!=self.prev else 0.0
        nv=max(self.val*(1+pos*ret)*(1-cost), 1.0)
        alpha_r=np.log(nv/self.val+1e-10)-np.log(1+ret+1e-10)
        self.val=nv; self.prev=pos; self.t+=1
        self.vals.append(nv); self.acts.append(pos)
        done=(self.t>=self.T-1)
        return (self._obs() if not done else np.zeros(15,np.float32)), float(alpha_r), done, False, {}
    def metrics(self):
        v=np.array(self.vals); r=np.diff(v)/v[:-1]
        sr=float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252))
        cr=float((v[-1]/v[0]-1)*100)
        pk=np.maximum.accumulate(v); mdd=float(abs(((v-pk)/pk).min())*100)
        return {"CR%":round(cr,2),"SR":round(sr,3),"MDD%":round(mdd,2)}

def make_env(S, R, rand=True, cost=True): return TradingEnv(S,R,rand,cost)

In [ ]:
# ── LSTM (PG/PPO) — temporal dependencies ─────────────────────
class LSTMPolicy(nn.Module):
    def __init__(self, hid=128, nl=2, na=3):
        super().__init__()
        self.hid=hid; self.nl=nl
        self.lstm=nn.LSTM(15,hid,nl,batch_first=True,dropout=0.1 if nl>1 else 0.0)
        self.fc=nn.Linear(hid,na); self.hidden=None
    def init_h(self, dev):
        self.hidden=(torch.zeros(self.nl,1,self.hid,device=dev),
                     torch.zeros(self.nl,1,self.hid,device=dev))
    def forward(self, x, reset=False):
        if reset or self.hidden is None: self.init_h(x.device)
        if x.dim()==1: x=x.unsqueeze(0).unsqueeze(0)
        out,self.hidden=self.lstm(x,self.hidden)
        self.hidden=(self.hidden[0].detach(), self.hidden[1].detach())
        return self.fc(out[:, -1, :]).squeeze(0)

class LSTMActorCritic(nn.Module):
    def __init__(self, hid=128, nl=2, na=3):
        super().__init__()
        self.hid=hid; self.nl=nl
        self.lstm=nn.LSTM(15,hid,nl,batch_first=True,dropout=0.1 if nl>1 else 0.0)
        self.actor=nn.Linear(hid,na); self.critic=nn.Linear(hid,1); self.hidden=None
    def init_h(self, dev):
        self.hidden=(torch.zeros(self.nl,1,self.hid,device=dev),
                     torch.zeros(self.nl,1,self.hid,device=dev))
    def forward(self, x, reset=False):
        if reset or self.hidden is None: self.init_h(x.device)
        if x.dim()==1: x=x.unsqueeze(0).unsqueeze(0)
        out,self.hidden=self.lstm(x,self.hidden)
        self.hidden=(self.hidden[0].detach(), self.hidden[1].detach())
        h=out[:, -1, :].squeeze(0)
        return self.actor(h), self.critic(h).squeeze()
    def forward_seq(self, x):
        x=x.unsqueeze(0); self.init_h(x.device)
        out,self.hidden=self.lstm(x,self.hidden)
        h=out.squeeze(0)
        return self.actor(h), self.critic(h)

# ── FFNN (DQL/DDPG) — replay buffer breaks temporal ordering ──
def make_ffnn(inp, hid, out, nl=2):
    lays=[nn.Linear(inp,hid),nn.LayerNorm(hid),nn.ReLU()]
    for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.LayerNorm(hid),nn.ReLU()]
    lays.append(nn.Linear(hid,out)); return nn.Sequential(*lays)

class QNet(nn.Module):
    def __init__(self, hid=256, nl=2):
        super().__init__(); self.net=make_ffnn(15,hid,3,nl)
    def forward(self,x): return self.net(x)

class DDPGActor(nn.Module):
    def __init__(self, hid=256, nl=2):
        super().__init__()
        lays=[nn.Linear(15,hid),nn.LayerNorm(hid),nn.ReLU()]
        for _ in range(nl-1): lays+=[nn.Linear(hid,hid),nn.LayerNorm(hid),nn.ReLU()]
        lays+=[nn.Linear(hid,1),nn.Tanh()]
        self.net=nn.Sequential(*lays)
    def forward(self,x): return self.net(x)
    def discrete(self, x):
        with torch.no_grad(): v=self.net(x.unsqueeze(0)).item()
        return (0 if v<-1/3 else (2 if v>1/3 else 1)), v

class DDPGCritic(nn.Module):
    def __init__(self, hid=256, nl=2):
        super().__init__(); self.net=make_ffnn(16,hid,1,nl)
    def forward(self,s,a): return self.net(torch.cat([s,a],dim=-1))

class ReplayBuffer:
    def __init__(self, cap=10000):
        self.cap=cap; self.ptr=0; self.sz=0
        self.S=np.zeros((cap,15),np.float32); self.A=np.zeros(cap,np.int64)
        self.R=np.zeros(cap,np.float32); self.NS=np.zeros((cap,15),np.float32)
        self.D=np.zeros(cap,np.float32); self.CA=np.zeros(cap,np.float32)
    def push(self,s,a,r,ns,d,ca=0.0):
        i=self.ptr
        self.S[i]=s; self.A[i]=a; self.R[i]=r; self.NS[i]=ns; self.D[i]=d; self.CA[i]=ca
        self.ptr=(i+1)%self.cap; self.sz=min(self.sz+1,self.cap)
    def sample(self,bs,dev):
        i=np.random.choice(self.sz,bs,replace=False)
        return (torch.FloatTensor(self.S[i]).to(dev), torch.LongTensor(self.A[i]).to(dev),
                torch.FloatTensor(self.R[i]).to(dev), torch.FloatTensor(self.NS[i]).to(dev),
                torch.FloatTensor(self.D[i]).to(dev), torch.FloatTensor(self.CA[i]).to(dev))
    def __len__(self): return self.sz

class OUNoise:
    def __init__(self, th=0.15, sig=0.2): self.th=th; self.sig=sig; self.reset()
    def reset(self): self.s=0.0
    def __call__(self): self.s+=self.th*(0-self.s)+self.sig*np.random.randn(); return self.s

In [ ]:
# ── 1. Policy Gradient (LSTM + Monte Carlo + entropy bonus) ───
class PGAgent:
    def __init__(self, hid=128, nl=2, lr=1e-3, gamma=0.99, ent=0.01):
        self.gamma=gamma; self.ent=ent
        self.policy=LSTMPolicy(hid,nl).to(DEVICE)
        self.opt=torch.optim.Adam(self.policy.parameters(),lr=lr,weight_decay=1e-4)
        self.lp=[]; self.rw=[]; self.en=[]
    def act(self, s, reset=False):
        lg=self.policy(torch.FloatTensor(s).to(DEVICE),reset)
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.lp.append(d.log_prob(a)); self.en.append(d.entropy()); return a.item()
    def store(self, r): self.rw.append(r)
    def update(self):
        if not self.rw: return {}
        G=0.0; ret=[]
        for r in reversed(self.rw): G=r+self.gamma*G; ret.insert(0,G)
        ret=torch.FloatTensor(ret).to(DEVICE)
        ret=(ret-ret.mean())/(ret.std()+1e-8)
        loss=-(torch.stack(self.lp)*ret).mean()-self.ent*torch.stack(self.en).mean()
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(),1.0)
        self.opt.step(); self.lp=[]; self.rw=[]; self.en=[]; return {}
    def greedy(self, s):
        with torch.no_grad(): return self.policy(torch.FloatTensor(s).to(DEVICE)).argmax().item()

# ── 2. PPO (LSTM Actor-Critic + GAE + clipped objective) ──────
class PPOAgent:
    def __init__(self, hid=128, nl=2, lr=3e-4, gamma=0.99,
                 lam=0.95, eps=0.2, vc=0.5, ec=0.01, epochs=4):
        self.gamma=gamma; self.lam=lam; self.eps=eps; self.vc=vc; self.ec=ec; self.epochs=epochs
        self.ac=LSTMActorCritic(hid,nl).to(DEVICE)
        self.opt=torch.optim.Adam(self.ac.parameters(),lr=lr,weight_decay=1e-4)
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]
    def act(self, s, reset=False):
        st=torch.FloatTensor(s).to(DEVICE)
        with torch.no_grad(): lg,val=self.ac(st,reset)
        d=torch.distributions.Categorical(logits=lg); a=d.sample()
        self.ss.append(s); self.aa.append(a.item())
        self.lp.append(d.log_prob(a).item()); self.vv.append(val.item()); return a.item()
    def store(self, r, done): self.rr.append(r); self.dd.append(done)
    def update(self, last_val=0.0):
        if len(self.rr)<2:
            self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return {}
        vals=self.vv+[last_val]; adv=[]; g=0.0
        for t in reversed(range(len(self.rr))):
            mask=1-self.dd[t]; delta=self.rr[t]+self.gamma*vals[t+1]*mask-vals[t]
            g=delta+self.gamma*self.lam*mask*g; adv.insert(0,g)
        adv=torch.FloatTensor(adv).to(DEVICE); adv=(adv-adv.mean())/(adv.std()+1e-8)
        ret=adv+torch.FloatTensor(self.vv).to(DEVICE)
        st=torch.FloatTensor(np.array(self.ss)).to(DEVICE)
        at=torch.LongTensor(self.aa).to(DEVICE); olp=torch.FloatTensor(self.lp).to(DEVICE)
        for _ in range(self.epochs):
            self.ac.hidden=None; lg,nv=self.ac.forward_seq(st)
            d=torch.distributions.Categorical(logits=lg); ratio=(d.log_prob(at)-olp).exp()
            al=-torch.min(ratio*adv,torch.clamp(ratio,1-self.eps,1+self.eps)*adv).mean()
            loss=al+self.vc*F.mse_loss(nv.squeeze(),ret)-self.ec*d.entropy().mean()
            self.opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(self.ac.parameters(),1.0); self.opt.step()
        self.ss=[]; self.aa=[]; self.lp=[]; self.rr=[]; self.vv=[]; self.dd=[]; return {}
    def greedy(self, s):
        with torch.no_grad(): lg,_=self.ac(torch.FloatTensor(s).to(DEVICE)); return lg.argmax().item()

# ── 3. DQL (FFNN + Double Q + experience replay) ──────────────
class DQLAgent:
    def __init__(self, hid=256, nl=2, lr=1e-3, gamma=0.99,
                 buf=10000, bs=64, tgt=100, eps_s=1.0, eps_e=0.05, eps_d=5000):
        self.gamma=gamma; self.bs=bs; self.tgt=tgt; self.steps=0
        self.eps=eps_s; self.eps_s=eps_s; self.eps_e=eps_e; self.eps_d=eps_d
        self.online=QNet(hid,nl).to(DEVICE); self.target=QNet(hid,nl).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict()); self.target.eval()
        self.opt=torch.optim.Adam(self.online.parameters(),lr=lr,weight_decay=1e-4)
        self.buf=ReplayBuffer(buf)
    def act(self, s):
        self.eps=max(self.eps_e,self.eps_s-self.steps/self.eps_d*(self.eps_s-self.eps_e))
        if np.random.rand()<self.eps: return np.random.randint(3)
        with torch.no_grad(): return self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE)).argmax().item()
    def store(self, s,a,r,ns,d): self.buf.push(s,a,r,ns,d); self.steps+=1
    def update(self):
        if len(self.buf)<self.bs: return {}
        s,a,r,ns,d,_=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.online(ns).argmax(1,keepdim=True); tq=self.target(ns).gather(1,na).squeeze()
            tgt=r+self.gamma*tq*(1-d)
        cq=self.online(s).gather(1,a.unsqueeze(1)).squeeze()
        loss=F.smooth_l1_loss(cq,tgt)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(),1.0); self.opt.step()
        if self.steps%self.tgt==0: self.target.load_state_dict(self.online.state_dict())
        return {}
    def greedy(self, s):
        with torch.no_grad(): return self.online(torch.FloatTensor(s).unsqueeze(0).to(DEVICE)).argmax().item()

# ── 4. DDPG (FFNN Actor-Critic + OU noise + soft updates) ─────
class DDPGAgent:
    def __init__(self, hid=256, nl=2, la=1e-4, lc=1e-3, gamma=0.99,
                 tau=0.005, buf=10000, bs=64, ou_t=0.15, ou_s=0.2):
        self.gamma=gamma; self.tau=tau; self.bs=bs
        self.actor=DDPGActor(hid,nl).to(DEVICE); self.critic=DDPGCritic(hid,nl).to(DEVICE)
        self.actor_t=DDPGActor(hid,nl).to(DEVICE); self.critic_t=DDPGCritic(hid,nl).to(DEVICE)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.critic_t.load_state_dict(self.critic.state_dict())
        self.actor_t.eval(); self.critic_t.eval()
        self.opt_a=torch.optim.Adam(self.actor.parameters(),lr=la,weight_decay=1e-4)
        self.opt_c=torch.optim.Adam(self.critic.parameters(),lr=lc,weight_decay=1e-4)
        self.buf=ReplayBuffer(buf); self.noise=OUNoise(ou_t,ou_s)
    def _soft(self,o,t):
        for p,pt in zip(o.parameters(),t.parameters()): pt.data.copy_(self.tau*p.data+(1-self.tau)*pt.data)
    def act(self, s, explore=True):
        st=torch.FloatTensor(s).to(DEVICE); disc,ca=self.actor.discrete(st)
        if explore: ca=float(np.clip(ca+self.noise(),-1,1)); disc=0 if ca<-1/3 else (2 if ca>1/3 else 1)
        return disc, ca
    def store(self,s,a,r,ns,d,ca): self.buf.push(s,a,r,ns,d,ca)
    def update(self):
        if len(self.buf)<self.bs: return {}
        s,a,r,ns,d,ca=self.buf.sample(self.bs,DEVICE)
        with torch.no_grad():
            na=self.actor_t(ns); tq=self.critic_t(ns,na).squeeze(); tgt=r+self.gamma*tq*(1-d)
        cq=self.critic(s,ca.unsqueeze(1)).squeeze(); cl=F.mse_loss(cq,tgt)
        self.opt_c.zero_grad(); cl.backward(); nn.utils.clip_grad_norm_(self.critic.parameters(),1.0); self.opt_c.step()
        pa=self.actor(s); al=-self.critic(s,pa).mean()
        self.opt_a.zero_grad(); al.backward(); nn.utils.clip_grad_norm_(self.actor.parameters(),1.0); self.opt_a.step()
        self._soft(self.actor,self.actor_t); self._soft(self.critic,self.critic_t); return {}
    def greedy(self, s): disc,_=self.act(s,explore=False); return disc

In [ ]:
def sharpe(vals):
    r=np.diff(np.array(vals))/np.array(vals[:-1])
    return float(np.mean(r)/(np.std(r)+1e-10)*np.sqrt(252)) if len(r)>1 else 0.0

def run_eval(agent, S, R):
    env=make_env(S,R,rand=False,cost=False); obs,_=env.reset()
    if hasattr(agent,"policy"): agent.policy.hidden=None
    if hasattr(agent,"ac"):     agent.ac.hidden=None
    done=False
    while not done: obs,_,done,_,_=env.step(agent.greedy(obs))
    return sharpe(env.vals), env.metrics()

def train_loop(agent, name, max_ep, eval_every, patience):
    best_sr=-np.inf; best_st=None; no_imp=0
    for ep in range(1,max_ep+1):
        env=make_env(train_S,train_R); obs,_=env.reset(); done=False; tot=0.0; lv=0.0
        if name=="PG":
            agent.policy.hidden=None; agent.act(obs,reset=True)
            while not done: a=agent.act(obs); obs,r,done,_,_=env.step(a); agent.store(r); tot+=r
            agent.update()
        elif name=="PPO":
            agent.ac.hidden=None
            while not done: a=agent.act(obs); obs,r,done,_,_=env.step(a); agent.store(r,done); tot+=r
            if not done:
                with torch.no_grad(): _,lv=agent.ac(torch.FloatTensor(obs).to(DEVICE)); lv=lv.item()
            agent.update(lv)
        elif name=="DQL":
            while not done: a=agent.act(obs); no,r,done,_,_=env.step(a); agent.store(obs,a,r,no,done); agent.update(); obs=no; tot+=r
        elif name=="DDPG":
            agent.noise.reset()
            while not done: a,ca=agent.act(obs); no,r,done,_,_=env.step(a); agent.store(obs,a,r,no,done,ca); agent.update(); obs=no; tot+=r

        if ep%eval_every==0:
            sr,m=run_eval(agent,val_S,val_R)
            print(f"  [{name}] ep={ep:4d} tot={tot:+.3f} val_SR={sr:.3f} val_CR={m['CR%']:.1f}%")
            if sr>best_sr:
                best_sr=sr; no_imp=0
                if   name=="PG":   best_st=copy.deepcopy(agent.policy.state_dict())
                elif name=="PPO":  best_st=copy.deepcopy(agent.ac.state_dict())
                elif name=="DQL":  best_st=copy.deepcopy(agent.online.state_dict())
                elif name=="DDPG": best_st=(copy.deepcopy(agent.actor.state_dict()), copy.deepcopy(agent.critic.state_dict()))
            else:
                no_imp+=1
                if no_imp>=patience: print(f"  [{name}] Early stop ep={ep}"); break
    if best_st:
        if   name=="PG":   agent.policy.load_state_dict(best_st)
        elif name=="PPO":  agent.ac.load_state_dict(best_st)
        elif name=="DQL":  agent.online.load_state_dict(best_st)
        elif name=="DDPG": agent.actor.load_state_dict(best_st[0]); agent.critic.load_state_dict(best_st[1])
    print(f"  [{name}] Best val SR={best_sr:.3f}"); return agent, best_sr

In [ ]:
print("="*60)
print(f"Training 4 agents on {ASSET} | Device: {DEVICE}")
print("="*60)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

trained={}; val_srs={}

cfgs = {
    "PG":   {"hid":128,"nl":2,"lr":1e-3,"gamma":0.99,"ent":0.01},
    "PPO":  {"hid":128,"nl":2,"lr":3e-4,"gamma":0.99,"lam":0.95,"eps":0.2},
    "DQL":  {"hid":256,"nl":2,"lr":1e-3,"gamma":0.99},
    "DDPG": {"hid":256,"nl":2,"la":1e-4,"lc":1e-3,"gamma":0.99,"tau":0.005},
}

for name,cfg in cfgs.items():
    msg = f"[{name}] Training..."
    print(msg)
    if   name=="PG":   agent=PGAgent(**cfg)
    elif name=="PPO":  agent=PPOAgent(**cfg)
    elif name=="DQL":  agent=DQLAgent(**cfg)
    elif name=="DDPG": agent=DDPGAgent(**cfg)
    agent,vsr=train_loop(agent,name,MAX_EP,EVAL_EVERY,PATIENCE)
    trained[name]=agent; val_srs[name]=vsr

best_name=max(val_srs,key=val_srs.get)
sep = "="*60
print(sep)
print("Val SR Summary:")
for n,v in val_srs.items():
    print(f"  {n}: {v:.3f}")
print(f"  Best by val SR: {best_name} ({val_srs[best_name]:.3f})")

In [ ]:
sep65 = "="*65
print("")
print(sep65)
print(f"FINAL BENCHMARK  {ASSET}  Test Set")
print(sep65)

def eval_on_test(agent, name):
    env=make_env(test_S,test_R,rand=False,cost=False); obs,_=env.reset()
    if hasattr(agent,"policy"): agent.policy.hidden=None
    if hasattr(agent,"ac"):     agent.ac.hidden=None
    done=False
    while not done: obs,_,done,_,_=env.step(agent.greedy(obs))
    return sharpe(env.vals), env.metrics()

bnh=make_env(test_S,test_R,rand=False,cost=False); bnh.reset()
for _ in range(bnh.T-1): bnh.step(2)
bsr=sharpe(bnh.vals); bm=bnh.metrics()
print(f"  {'B&H':8s} | CR={bm['CR%']:6.1f}% SR={bm['SR']:6.3f} MDD={bm['MDD%']:5.1f}%")

rows=[{"Model":"B&H",**bm,"ValSR":"--"}]
for name,agent in trained.items():
    tsr,m=eval_on_test(agent,name)
    beat="OK" if m["CR%"]>bm["CR%"] else "--"
    print(f"[{beat}] {name:8s} | CR={m['CR%']:6.1f}% SR={m['SR']:6.3f} MDD={m['MDD%']:5.1f}% | ValSR={val_srs[name]:.3f}")
    rows.append({"Model":name,**m,"ValSR":round(val_srs[name],3)})

df_res=pd.DataFrame(rows)
df_res.to_csv(f"results_{ASSET}.csv",index=False)
print(f"Saved results_{ASSET}.csv")
print(df_res.to_string(index=False))

import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(14,7))
ax.plot(bnh.vals,label="Buy & Hold",color="black",linestyle="--",lw=2)
colors={"PG":"#1f77b4","PPO":"#ff7f0e","DQL":"#2ca02c","DDPG":"#d62728"}
for name,agent in trained.items():
    env=make_env(test_S,test_R,rand=False,cost=False); obs,_=env.reset()
    if hasattr(agent,"policy"): agent.policy.hidden=None
    if hasattr(agent,"ac"):     agent.ac.hidden=None
    done=False
    while not done: obs,_,done,_,_=env.step(agent.greedy(obs))
    ax.plot(env.vals,label=name,color=colors.get(name,"gray"),lw=1.5)
ax.set_title(f"{ASSET} Portfolio Value - Test Set",fontsize=14)
ax.set_xlabel("Days"); ax.set_ylabel("Value ($)")
ax.legend(fontsize=11); ax.grid(True,alpha=0.3); plt.tight_layout()
plt.savefig(f"portfolio_{ASSET}.png",dpi=150,bbox_inches="tight"); plt.close()
print(f"Saved portfolio_{ASSET}.png")